# Lab 08 — Sequence architectures and forecasting protocol

**Dataset:** [huggingface/electricity-production](https://huggingface.co/datasets/huggingface/electricity-production), a compact monthly electricity-production time series stored as Parquet. The main experiment is about **evaluation discipline**: chronological split versus random split, then one-step versus recursive multi-step forecasting.

## The math behind the experiment

For a one-step forecast, the model estimates

$$
\hat{y}_{t+1}=f_\theta(x_{t-k+1:t}).
$$

A common regression loss is

$$
MSE=\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2.
$$

The critical experimental constraint is chronological ordering: training examples must not contain information from the future evaluation period. A random split can make a forecasting system look better while quietly leaking temporal information.

In [ ]:
!pip -q install datasets scikit-learn pandas numpy torch matplotlib
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler
import numpy as np,pandas as pd

raw=load_dataset("huggingface/electricity-production",split="train")
raw_values=np.asarray(raw["IPG2211A2N"],dtype=np.float32)
raw_values=raw_values[np.isfinite(raw_values)]

# Define the temporal cutoff before fitting any preprocessing.
cutoff=int(.8*len(raw_values))
scaler=StandardScaler()
scaler.fit(raw_values[:cutoff].reshape(-1,1))  # training period only
values=scaler.transform(raw_values.reshape(-1,1)).ravel()

T=24
target_indices=np.arange(T,len(values))
X=np.array([values[t-T:t] for t in target_indices],dtype=np.float32)
y=values[target_indices].astype(np.float32)
train_mask=target_indices<cutoff
test_mask=target_indices>=cutoff
Xtr,ytr=X[train_mask],y[train_mask]
Xte,yte=X[test_mask],y[test_mask]

assert target_indices[train_mask].max()<cutoff
assert target_indices[test_mask].min()>=cutoff
assert len(Xtr)>0 and len(Xte)>0
print("Monthly electricity-production series:",raw_values.shape)
print("Training windows / test windows:",Xtr.shape,Xte.shape)
print("Last training target index:",target_indices[train_mask].max())
print("First test target index:",target_indices[test_mask].min())
print("Scaler fit indices: 0 through",cutoff-1,"(training period only)")

## Step 1 — Baseline

Use the last observed value as a naive forecast. A neural model should beat this baseline to justify its complexity.

In [ ]:
from sklearn.metrics import mean_squared_error
naive=np.repeat(Xte[:,-1,None],1,axis=1).ravel()
print("chronological naive MSE",mean_squared_error(yte,naive))

# Intentionally invalid random split of overlapping windows. This is a diagnostic
# contrast, not the deployment-valid headline metric.
rng=np.random.default_rng(4)
idx=rng.permutation(len(X))
cut=int(.8*len(X))
tr,va=idx[:cut],idx[cut:]
rand_naive=np.repeat(X[va][:,-1,None],1,axis=1).ravel()
print("random-split naive MSE",mean_squared_error(y[va],rand_naive))
print("Reminder: random-window scores are not comparable to the future-only test score.")

## Step 2 — Leakage experiment

There are two distinct leakage risks here:

1. **Preprocessing leakage:** fitting a scaler on the entire series lets future distribution information influence the training representation. The setup cell avoids this by fitting `StandardScaler` only on the training period.
2. **Split leakage:** randomly dividing heavily overlapping windows lets near-duplicate contexts appear on both sides. The random split below is intentionally invalid and is shown only to illustrate how a score can mislead.

Use the chronological future holdout as the primary result. Do not assume the random-split score must always be lower; the lesson is that it answers a different question and cannot estimate future deployment performance.

In [ ]:
from sklearn.linear_model import Ridge
# Primary result: fit on past targets, evaluate on future targets.
m=Ridge(alpha=1.0).fit(Xtr,ytr)
chron=mean_squared_error(yte,m.predict(Xte))

# Invalid random-window comparison. Keep it visibly separate from the primary result.
random_model=Ridge(alpha=1.0).fit(X[tr],y[tr])
rand=mean_squared_error(y[va],random_model.predict(X[va]))
pd.DataFrame([
    ["chronological future holdout",chron,"Primary deployment-aligned estimate"],
    ["random overlapping windows",rand,"Invalid for future forecasting; diagnostic only"]
],columns=["evaluation_protocol","MSE","interpretation"])

## Step 3 — Recursive horizon

For a one-step model, recursively feed its own prediction back into the input window. Record error at horizons 1, 2, 4 and 8. Error accumulation is the mechanism to explain.

In [ ]:
x=Xte[0].copy(); rows=[]
for h in [1,2,4,8]:
 cur=x.copy(); preds=[]
 for _ in range(h):
  p=float(m.predict(cur.reshape(1,-1))[0]); preds.append(p); cur=np.r_[cur[1:],p]
 true=yte[:h]; rows.append([h,mean_squared_error(true,preds)])
pd.DataFrame(rows,columns=["horizon","recursive_MSE"])

## Visualize the evaluation contract — time order and forecast horizon

The first graph compares a deployment-aligned chronological holdout with an intentionally invalid random split of overlapping windows. The second shows recursive error at selected horizons. The scaler is fitted on training-period values only; test windows may use earlier historical observations because those would be available at the forecast origin.

In [ ]:
import matplotlib.pyplot as plt

protocol_results = pd.DataFrame(
    [["chronological", chron], ["random split", rand]],
    columns=["protocol", "MSE"]
)
horizon_results = pd.DataFrame(rows, columns=["horizon", "recursive_MSE"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(protocol_results["protocol"], protocol_results["MSE"])
axes[0].set_title("Evaluation protocol matters")
axes[0].set_ylabel("MSE (lower is better)")
axes[0].grid(axis="y", alpha=.25)

axes[1].plot(horizon_results["horizon"], horizon_results["recursive_MSE"], marker="o")
axes[1].set_title("Recursive forecast error")
axes[1].set_xlabel("Forecast horizon (steps)")
axes[1].set_ylabel("MSE")
axes[1].grid(alpha=.25)
fig.tight_layout()
plt.show()

## Conclusion / answer key

Expected: chronological evaluation is the defensible primary result; random splitting can produce misleading evidence when temporal dependence exists. Recursive forecasting should generally accumulate error with horizon.

### Research extension
Compare Elman/Jordan-style state choices at matched parameter count and evaluate both one-step and multi-step error.